# SemDiD Smoke Test — Colab Free Tier

**Goal:** verify that SemDiD (Semantic-guided Diverse Decoding, NeurIPS 2025) runs end-to-end on Colab's free T4 GPU with `Qwen/Qwen2.5-0.5B-Instruct`, before committing to the full NLP A3 project plan.

**Success criterion:** cell-by-cell runs to completion, and the final cell prints three visibly different feedback variants for a toy student essay.

**Before running:** `Runtime → Change runtime type → T4 GPU`. Also ensure you have ~14 GB free VRAM (fresh runtime).

## 1. Confirm GPU is attached

In [ ]:
import torch
assert torch.cuda.is_available(), 'No CUDA device. Switch runtime to GPU (T4) and rerun.'
props = torch.cuda.get_device_properties(0)
print(f'Device:  {props.name}')
print(f'Memory:  {props.total_memory / 1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA:    {torch.version.cuda}')

## 2. Install dependencies

This cell takes **10–15 minutes** the first time (VLLM compiles CUDA extensions). Go make coffee.

We skip DeepSpeed and lm-eval-harness — we don't need them for the smoke test or our custom-prompt use case.

In [ ]:
# Pin vllm + transformers + tokenizers to a known-good combo from the SemDiD
# release window (Feb 2025). Doing it as a single resolver pass avoids the
# 'vllm wants new tokenizers but transformers 4.49 wants old tokenizers' fight.
%pip install -q \
    'vllm==0.7.3' \
    'transformers==4.49.0' \
    'tokenizers>=0.21,<0.22' \
    sentence-transformers accelerate

# Colab's base image ships torchcodec built against a newer torch ABI than
# vllm 0.7.3 pins. transformers auto-imports torchcodec, so the mismatch
# surfaces as a cryptic 'libavutil.so.60 not found' at any import site.
# We don't need video/audio decoding for text generation — rip it out.
%pip uninstall -y torchcodec

print('Dependencies installed. IMPORTANT: restart runtime now — Runtime -> Restart session — then resume from the next cell.')


### Restart the runtime before continuing

`Runtime -> Restart session` (or press the button above). This is required after upgrading `transformers`, otherwise Python still has the old version in memory and you'll hit weird import errors later.

In [ ]:
# Run this AFTER restarting the runtime, before continuing. It confirms the
# install stuck and the versions are the ones SemDiD was written against.
import importlib.metadata as md
for pkg in ('vllm', 'transformers', 'tokenizers', 'sentence-transformers', 'torch'):
    try:
        print(f'{pkg:<22} {md.version(pkg)}')
    except md.PackageNotFoundError:
        print(f'{pkg:<22} NOT INSTALLED')

try:
    md.version('torchcodec')
    print('\nWARNING: torchcodec is still installed; uninstall it or imports will fail.')
except md.PackageNotFoundError:
    print('\ntorchcodec is uninstalled (good — it was ABI-incompatible with our pinned torch).')

# Sanity-check that OffloadedCache lives where SemDiD expects it.
from transformers.cache_utils import OffloadedCache  # should not raise
print('OffloadedCache import works — transformers version is compatible.')


## 3. Clone SemDiD

In [ ]:
import os
if not os.path.exists('/content/SemDiD'):
    !git clone --depth 1 https://github.com/bhutley/SemDiD.git /content/SemDiD
else:
    print('SemDiD already present, skipping clone.')
!ls /content/SemDiD/my_lm_eval/lm_eval/models/semantic_search.py

## 4. Apply the monkey-patch

SemDiD intercepts HuggingFace's `_group_beam_search` method. After the patch, any HF model with `model.use_semantic_diverse_beam_search = True` and `num_beam_groups > 1` will use SemDiD.

In [ ]:
# IMPORTANT: do NOT add `/content/SemDiD/my_lm_eval/lm_eval/models` to
# sys.path. That directory contains a file named `gguf.py` which shadows the
# real PyPI `gguf` package (used by VLLM for quantization file support).
# VLLM's `import gguf` would then find the wrong file and fail.
#
# Instead, load semantic_search.py as a standalone module under a unique
# name via importlib. No sys.path pollution, no accidental shadowing.
import importlib.util

SPEC_PATH = '/content/SemDiD/my_lm_eval/lm_eval/models/semantic_search.py'
spec = importlib.util.spec_from_file_location('semdid_patch', SPEC_PATH)
semdid_patch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(semdid_patch)

semdid_patch.patch_transformers_generation()
print('Monkey-patch applied (via importlib, no sys.path pollution).')


## 5. Load Qwen-0.5B-Instruct (HuggingFace side)

NOTE: SemDiD internally *also* loads the model via VLLM (that's how the algorithm works). Both copies sit in GPU memory simultaneously. For Qwen-0.5B on a T4 this is fine — the two copies are ~2 GB combined.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig
import torch

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map='auto',
    torch_dtype=torch.float16,  # T4 (compute 7.5) lacks bf16 support; use fp16
    trust_remote_code=True,
)
model.use_semantic_diverse_beam_search = True   # flip the flag SemDiD checks
print(f'Loaded {MODEL_NAME} on {model.device}')

## 6. Build a toy feedback prompt

A short, deliberately mediocre student essay — we want the model to have plenty of things to comment on (organisation, argument, evidence, mechanics).

In [ ]:
student_essay = '''
Technology has changed the way we live alot. Many people think phones are bad
but actually they help us communicate with family members who live far away.
Some students use phones in class which is very distracting to other people.
Also social media can be addicting. Overall I think technology is good because
without it we would be stuck in the stone age and thats not good for anyone.
'''

system = 'You are an experienced writing teacher providing constructive feedback to a student.'
user = (
    'Please give feedback on the following student essay. '
    'Comment on argument development, evidence, organisation, and mechanics. '
    'Write a single concise feedback paragraph.\n\n'
    f'Essay:\n{student_essay}\n\nFeedback:'
)

messages = [
    {'role': 'system', 'content': system},
    {'role': 'user',   'content': user},
]
input_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
input_ids = tokenizer(input_text, return_tensors='pt').input_ids.to(model.device)
print(f'Prompt length: {input_ids.shape[1]} tokens')

## 7. Generate 3 feedback variants with SemDiD

Config notes:
- `num_beams=9, num_beam_groups=3` → 3 diverse groups of 3 beams each.
- `forward_steps=300` is the SemDiD-specific lookahead horizon (the expensive bit).
- `gpu_memory_utilization=0.5` leaves headroom for the HuggingFace copy of the model.
- Expect **30–90 seconds** for this cell on T4.

In [ ]:
import time

# The `vllm_model_name` / `gpu_memory_utilization` / `embedding_model_name`
# kwargs shown in SemDiD's example_usage are dead — the real function
# hardcodes them internally (see lines 985-987 of semantic_search.py).
# So don't pass them; they'd just be rejected by _validate_model_kwargs.
gen_config = GenerationConfig(
    max_length=input_ids.shape[1] + 300,
    num_beams=9,
    num_beam_groups=3,
    diversity_penalty=1.0,
    repetition_penalty=1.2,
    forward_steps=300,             # SemDiD-specific lookahead
    num_return_sequences=3,
    pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id,
)

start = time.time()
outputs = model.generate(input_ids, generation_config=gen_config)
elapsed = time.time() - start
print(f'Generation took {elapsed:.1f}s\n')

# SemDiD's patched path returns a list of decoded strings; stock HF returns
# a tensor. Handle both so this cell also works as a sanity check against
# the un-patched baseline later.
if isinstance(outputs, list) and all(isinstance(x, str) for x in outputs):
    feedback_variants = [s.strip() for s in outputs]
else:
    feedback_variants = [
        tokenizer.decode(out[input_ids.shape[1]:], skip_special_tokens=True).strip()
        for out in outputs
    ]

for i, text in enumerate(feedback_variants):
    print(f'=== Feedback variant {i + 1} ===')
    print(text)
    print()


## 8. Quick diversity sanity check

Compute pairwise cosine similarity of the three feedback variants in sentence-embedding space. Lower mean similarity = more semantically distinct. If SemDiD is working, we'd expect mean off-diagonal similarity noticeably below 1.0 — and lower than what plain temperature sampling would produce (we'll compare formally later).

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

enc = SentenceTransformer('all-MiniLM-L6-v2')
embs = enc.encode(feedback_variants, convert_to_numpy=True, normalize_embeddings=True)
sim = embs @ embs.T

print('Pairwise cosine similarity matrix:')
print(np.round(sim, 3))

n = len(feedback_variants)
offdiag = (sim.sum() - n) / (n * (n - 1))
print(f'\nMean off-diagonal similarity: {offdiag:.3f}  (lower = more diverse)')
print('Rule of thumb for this smoke test: <0.85 is a good sign, >0.95 means we may be generating near-duplicates.')

## What success looks like

All cells run without raising. Cell 7 prints three distinguishable feedback paragraphs (different wording, ideally different areas of focus). Cell 8 reports mean off-diagonal similarity well below 1.0.

## Common failures and what they mean

- **`No CUDA device`** — runtime isn't GPU. `Runtime → Change runtime type → T4`.
- **VLLM install fails / flashinfer build errors** — Colab's CUDA version may have drifted. Try a pinned `vllm==0.6.x`, or switch to Kaggle.
- **OOM during cell 7** — reduce `gpu_memory_utilization` or `forward_steps`, or drop `num_beams` to 6.
- **Generation completes but all three variants are identical** — the monkey-patch didn't take effect. Verify `patch_transformers_generation()` ran *before* `model.generate`, and that `model.use_semantic_diverse_beam_search` is `True`.
- **Total time > 5 min on T4** — expected on first run (VLLM model load); subsequent generations in the same session should be faster.

If you hit anything weird, copy the error traceback back and we'll diagnose.